# gTCN evaluation: two effects and averaged model

This notebook evaluates `best_BritPlexi.pt` and `best_FenderDouble.pt` on the same dry guitar clip. It compares dry input, wet ground truth, and wet prediction for both effects. It then creates a third model by averaging the compatible floating-point parameters of the two checkpoints and evaluates that averaged model.

The final section lets you provide a new dry guitar WAV file.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
import torchaudio
from IPython.display import Audio, display

from dataset import GuitarPairIndex
from inference import GuitarRenderer
from model import GTCNModel
from utils import resolve_path

print("Repo root:", REPO_ROOT)
print("CUDA available:", torch.cuda.is_available())

## Load both effect models

In [ ]:
TIMBRES = ["BritPlexi", "FenderDouble"]
CHECKPOINTS = {
    timbre: REPO_ROOT / "checkpoints" / f"best_{timbre}.pt"
    for timbre in TIMBRES
}
RENDERERS = {
    timbre: GuitarRenderer(str(checkpoint_path))
    for timbre, checkpoint_path in CHECKPOINTS.items()
}
SAMPLE_RATE = next(iter(RENDERERS.values())).sample_rate

for timbre, checkpoint_path in CHECKPOINTS.items():
    print(f"{timbre}: {checkpoint_path}")
print("Sample rate:", SAMPLE_RATE)

## Helper functions

In [ ]:
def load_audio(path, target_sr=SAMPLE_RATE):
    audio, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=-1)
    if sr != target_sr:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, target_sr
        ).squeeze(0).numpy()
    return audio


@torch.no_grad()
def render_with_model(model, dry_audio, dry_ref, wet_ref, device):
    dry_t = torch.from_numpy(dry_audio).float().to(device).view(1, 1, -1)
    dry_ref_t = torch.from_numpy(dry_ref).float().to(device).view(1, 1, -1)
    wet_ref_t = torch.from_numpy(wet_ref).float().to(device).view(1, 1, -1)
    predicted, _ = model(dry_t, dry_ref_t, wet_ref_t)
    return predicted.squeeze().detach().cpu().numpy()


@torch.no_grad()
def render_effect(timbre, dry_audio):
    renderer = RENDERERS[timbre]
    dry_ref, wet_ref = renderer.sample_reference(timbre)
    return render_with_model(renderer.model, dry_audio, dry_ref, wet_ref, renderer.device)


def plot_waveforms(signals, labels, max_seconds=5.0):
    n = int(SAMPLE_RATE * max_seconds)
    fig, axes = plt.subplots(len(signals), 1, figsize=(12, 2.1 * len(signals)), sharex=True)
    axes = np.atleast_1d(axes)
    for axis, signal, label in zip(axes, signals, labels):
        clipped = signal[:n]
        axis.plot(np.arange(len(clipped)) / SAMPLE_RATE, clipped, linewidth=0.6)
        axis.set_title(label)
        axis.set_ylim(-1.05, 1.05)
    axes[-1].set_xlabel("Time (s)")
    fig.tight_layout()
    plt.show()


def plot_spectrograms(signals, labels):
    n_fft, hop = 2048, 512
    window = torch.hann_window(n_fft)
    fig, axes = plt.subplots(1, len(signals), figsize=(5 * len(signals), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for axis, signal, label in zip(axes, signals, labels):
        spec = torch.stft(torch.from_numpy(signal).float(), n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        db = 20 * torch.log10(spec.abs().clamp(min=1e-5))
        image = axis.imshow(db.numpy(), origin="lower", aspect="auto", extent=[0, len(signal) / SAMPLE_RATE, 0, SAMPLE_RATE / 2], cmap="magma")
        axis.set_title(label)
        axis.set_xlabel("Time (s)")
    axes[0].set_ylabel("Frequency (Hz)")
    fig.colorbar(image, ax=axes, fraction=0.02, pad=0.02, label="dB")
    plt.show()


def compare_metrics(predicted, target):
    length = min(len(predicted), len(target))
    predicted, target = predicted[:length], target[:length]
    print(f"L1: {np.mean(np.abs(predicted - target)):.4f}   RMSE: {np.sqrt(np.mean((predicted - target) ** 2)):.4f}")

## Compare both effects on one matching dry clip

In [ ]:
data_cfg = next(iter(RENDERERS.values())).cfg["data"]
x_root = resolve_path(data_cfg["x_root"])
y_root = resolve_path(data_cfg["y_root"])
pair_indices = {timbre: GuitarPairIndex(x_root, y_root, timbre) for timbre in TIMBRES}

EXAMPLE_IDX = 0
brit_pair = sorted(pair_indices["BritPlexi"].pairs)[EXAMPLE_IDX]
dry_path, brit_wet_path, _ = brit_pair
double_by_dry = {pair[0]: pair for pair in pair_indices["FenderDouble"].pairs}
_, double_wet_path, _ = double_by_dry[dry_path]

dry_audio = load_audio(dry_path)
wet_truth = {
    "BritPlexi": load_audio(brit_wet_path),
    "FenderDouble": load_audio(double_wet_path),
}
wet_predicted = {timbre: render_effect(timbre, dry_audio) for timbre in TIMBRES}

print("Dry:", dry_path)
for timbre in TIMBRES:
    print(f"{timbre} truth:", wet_truth[timbre].shape, "prediction:", wet_predicted[timbre].shape)
    compare_metrics(wet_predicted[timbre], wet_truth[timbre])

In [ ]:
plot_spectrograms(
    [dry_audio, wet_truth["BritPlexi"], wet_predicted["BritPlexi"]],
    ["Dry", "BritPlexi - wet truth", "BritPlexi - wet predicted"],
)
plot_spectrograms(
    [dry_audio, wet_truth["FenderDouble"], wet_predicted["FenderDouble"]],
    ["Dry", "FenderDouble - wet truth", "FenderDouble - wet predicted"],
)

In [ ]:
print("Dry (input):")
display(Audio(dry_audio, rate=SAMPLE_RATE))
for timbre in TIMBRES:
    print(f"{timbre} - wet truth:")
    display(Audio(wet_truth[timbre], rate=SAMPLE_RATE))
    print(f"{timbre} - wet predicted:")
    display(Audio(wet_predicted[timbre], rate=SAMPLE_RATE))

In [ ]:
plot_waveforms(
    [
        dry_audio,
        wet_truth["BritPlexi"], wet_predicted["BritPlexi"],
        wet_truth["FenderDouble"], wet_predicted["FenderDouble"],
    ],
    [
        "Dry", "BritPlexi - wet truth", "BritPlexi - wet predicted",
        "FenderDouble - wet truth", "FenderDouble - wet predicted",
    ],
)

## Average the BritPlexi and FenderDouble parameters

In [ ]:
brit_checkpoint = torch.load(str(CHECKPOINTS["BritPlexi"]), map_location="cpu")
double_checkpoint = torch.load(str(CHECKPOINTS["FenderDouble"]), map_location="cpu")
brit_state = brit_checkpoint["model_state_dict"]
double_state = double_checkpoint["model_state_dict"]

if set(brit_state) != set(double_state):
    raise ValueError("Checkpoint parameter keys differ.")
averaged_state = {}
for name, brit_value in brit_state.items():
    double_value = double_state[name]
    if brit_value.shape != double_value.shape:
        raise ValueError(f"Shape mismatch for {name}")
    averaged_state[name] = (brit_value + double_value) / 2 if torch.is_floating_point(brit_value) else brit_value

averaged_model = GTCNModel.from_config(brit_checkpoint["config"]["model"]).eval()
averaged_model.load_state_dict(averaged_state)
print(f"Averaged {len(averaged_state)} parameter/buffer tensors.")

In [ ]:
# Use a mixed reference for the averaged model: same dry reference and
# arithmetic mean of the two corresponding wet reference signals.
brit_renderer = RENDERERS["BritPlexi"]
double_renderer = RENDERERS["FenderDouble"]
brit_dry_ref, brit_wet_ref = brit_renderer.sample_reference("BritPlexi")
double_dry_ref, double_wet_ref = double_renderer.sample_reference("FenderDouble")
reference_length = min(len(brit_dry_ref), len(double_dry_ref), len(brit_wet_ref), len(double_wet_ref))
mixed_dry_ref = ((brit_dry_ref[:reference_length] + double_dry_ref[:reference_length]) / 2).astype(np.float32)
mixed_wet_ref = ((brit_wet_ref[:reference_length] + double_wet_ref[:reference_length]) / 2).astype(np.float32)

averaged_wet_predicted = render_with_model(
    averaged_model,
    dry_audio,
    mixed_dry_ref,
    mixed_wet_ref,
    device="cpu",
)
print("Averaged-model output:", averaged_wet_predicted.shape)

In [ ]:
plot_spectrograms(
    [dry_audio, averaged_wet_predicted],
    ["Dry", "Averaged BritPlexi + FenderDouble - wet predicted"],
)

print("Dry (input):")
display(Audio(dry_audio, rate=SAMPLE_RATE))
print("Averaged model - wet predicted:")
display(Audio(averaged_wet_predicted, rate=SAMPLE_RATE))

## Try a new dry guitar recording

In [ ]:
# Change this path to your own dry guitar WAV file.
NEW_DRY_PATH = REPO_ROOT.parent / "GTtimbre" / "data" / "X" / "GOAT" / "item_5.wav"
new_dry_audio = load_audio(NEW_DRY_PATH)
new_wet_predicted = render_effect("BritPlexi", new_dry_audio)

print(f"Loaded {NEW_DRY_PATH}, {len(new_dry_audio) / SAMPLE_RATE:.2f}s")
display(Audio(new_dry_audio, rate=SAMPLE_RATE))
display(Audio(new_wet_predicted, rate=SAMPLE_RATE))

In [ ]:
output_dir = REPO_ROOT / "outputs" / "eval"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{Path(NEW_DRY_PATH).stem}-BritPlexi-predicted.wav"
sf.write(str(output_path), new_wet_predicted, SAMPLE_RATE)
print("Saved:", output_path)